# CLIP Backbone Verification

**Model**: OpenAI CLIP — multiple sizes on HuggingFace  
**Arch**: ViT image encoder + Transformer text encoder (contrastive training)  
**Embed dim**: 512 (ViT-B/32, ViT-B/16) · 768 (ViT-L/14, ViT-L/14@336)  
**Access**: HuggingFace, no auth  
**Zero-shot**: Yes — paired text encoder

Paper: [Learning Transferable Visual Models from Natural Language Supervision (ICML 2021)](https://arxiv.org/abs/2103.00020)

Available variants:

| Hub ID | Arch | Input | Embed dim |
|---|---|---|---|
| `openai/clip-vit-base-patch32` | ViT-B/32 | 224 | 512 |
| `openai/clip-vit-base-patch16` | ViT-B/16 | 224 | 512 |
| `openai/clip-vit-large-patch14` | ViT-L/14 | 224 | 768 |
| `openai/clip-vit-large-patch14-336` | ViT-L/14 | 336 | 768 |

**Context**: Vanilla CLIP is the natural baseline for all medical CLIP variants
(BiomedCLIP, CheXzero, PubMedCLIP, MedSigLIP). Running it establishes how much
domain-specific training actually helps.

In [ ]:
%pip install -q transformers torch torchvision Pillow

In [ ]:
import os

# Control which GPU(s) are visible to this notebook.
# Set CUDA_VISIBLE_DEVICES in your shell before launching Jupyter to override,
# e.g.:  CUDA_VISIBLE_DEVICES=2 jupyter lab
# Or change the default value below.
os.environ.setdefault("CUDA_VISIBLE_DEVICES", "0")
print(f"CUDA_VISIBLE_DEVICES = {os.environ['CUDA_VISIBLE_DEVICES']}")

## 1. Load model

In [ ]:
import torch
from transformers import CLIPModel, CLIPProcessor

# Pick a variant — ViT-L/14 is the standard strong baseline
HUB = "openai/clip-vit-large-patch14"   # or clip-vit-base-patch32 / -patch16 / -large-patch14-336

EMBED_DIMS = {
    "openai/clip-vit-base-patch32":         512,
    "openai/clip-vit-base-patch16":         512,
    "openai/clip-vit-large-patch14":        768,
    "openai/clip-vit-large-patch14-336":    768,
}

processor = CLIPProcessor.from_pretrained(HUB)
model     = CLIPModel.from_pretrained(HUB)
model.eval()

print("Hub:", HUB)
print("Projection dim:", model.config.projection_dim)              # 768 for ViT-L
print("Vision hidden size:", model.config.vision_config.hidden_size)  # 1024 (ViT-L internal)
print("Image size:", model.config.vision_config.image_size)        # 224

## 2. Preprocess image

In [ ]:
from PIL import Image
import numpy as np

def preprocess(path: str) -> dict:
    img = Image.open(path).convert("RGB")   # CLIP always expects RGB
    return processor(images=img, return_tensors="pt")

# CLIP normalization — different from ImageNet and SigLIP
print("Image mean:", processor.image_processor.image_mean)
# (0.48145466, 0.4578275, 0.40821073)
print("Image std: ", processor.image_processor.image_std)
# (0.26862954, 0.26130258, 0.27577711)

dummy_pil = Image.fromarray(np.ones((512, 512, 3), dtype=np.uint8) * 128)
inputs    = processor(images=dummy_pil, return_tensors="pt")
print("pixel_values:", inputs["pixel_values"].shape)   # [1, 3, 224, 224]

## 3. Extract image embeddings

In [ ]:
import torch.nn.functional as F

with torch.no_grad():
    # Option A: projected joint-space features (use for zero-shot)
    img_feats = model.get_image_features(pixel_values=inputs["pixel_values"])  # [B, 512/768]

    # Option B: raw ViT CLS token before projection (larger dim, better for linear probe)
    vis_out   = model.vision_model(pixel_values=inputs["pixel_values"])
    raw_cls   = vis_out.pooler_output   # [B, 1024 for ViT-L] — BEFORE linear projection

img_norm = F.normalize(img_feats, dim=-1)   # L2-normalize for cosine sim

print("get_image_features (projected):", img_feats.shape)  # [1, 768] for ViT-L
print("vision_model CLS (raw):        ", raw_cls.shape)    # [1, 1024] for ViT-L
print("norm of projected:             ", img_feats.norm(dim=-1).item())

## 4. Zero-shot: text embeddings & similarity

In [ ]:
prompts = [
    "A chest X-ray with no acute findings.",
    "Cardiomegaly seen on chest radiograph.",
    "Bilateral pleural effusions.",
    "Right lower lobe pneumonia.",
    "Left-sided pneumothorax.",
]

text_inputs = processor(text=prompts, return_tensors="pt",
                        padding=True, truncation=True)

with torch.no_grad():
    text_feats = model.get_text_features(**text_inputs)   # [5, 768]
    text_norm  = F.normalize(text_feats, dim=-1)

# Scale by learned logit_scale (CLIP uses softmax temperature)
logit_scale = model.logit_scale.exp()
logits = logit_scale * (img_norm @ text_norm.T).squeeze(0)  # [5]
probs  = logits.softmax(dim=0)

for p, s, prob in zip(prompts, logits.tolist(), probs.tolist()):
    print(f"{prob:.3f}  {s:6.2f}  {p}")

## 5. Multi-variant benchmark

In [ ]:
variants = [
    "openai/clip-vit-base-patch32",
    "openai/clip-vit-base-patch16",
    "openai/clip-vit-large-patch14",
]

for hub in variants:
    proc = CLIPProcessor.from_pretrained(hub)
    mdl  = CLIPModel.from_pretrained(hub).eval()
    inp  = proc(images=dummy_pil, return_tensors="pt")
    with torch.no_grad():
        feats = mdl.get_image_features(**inp)   # [1, proj_dim]
        raw   = mdl.vision_model(**inp).pooler_output
    print(f"{hub.split('/')[-1]}: proj={feats.shape}  raw_cls={raw.shape}")

## 6. Projected vs raw CLS for linear probe

For classification linear probes, the **raw ViT CLS token** (before the linear
projection head) often outperforms the projected joint-space embedding, because
the projection discards classification-relevant information to align with text.

In [ ]:
# Quick cosine similarity between projected and raw CLS
proj_norm = F.normalize(img_feats, dim=-1)          # [1, 768]
raw_norm  = F.normalize(raw_cls,   dim=-1)          # [1, 1024]

# Can't compare directly (different dims) — just note dims for downstream use
print("Projected (joint space): ", img_feats.shape, "← use for zero-shot")
print("Raw CLS (larger dim):    ", raw_cls.shape,   "← try for linear probe")

## Segmentation mode — patch tokens

For dense prediction, extract the per-patch token sequence from `vision_model.last_hidden_state`.
Skip CLS (index 0) and reshape to a 2-D spatial grid. For ViT-L/14 at 224×224: **16×16** patch
grid, 1024-dim (pre-projection) features.

In [ ]:
import math

# Extract patch tokens from the vision tower's last hidden state.
# vision_model returns [B, 1+N_patches, D_vision] where D_vision=1024 for ViT-L/14.
with torch.no_grad():
    vis = model.vision_model(pixel_values=pv)

patches = vis.last_hidden_state[:, 1:]   # [B, N_patches, D_vision] — skip CLS

B, N, D = patches.shape
H = W = int(math.sqrt(N))   # 16 for ViT-L/14 at 224px (224/14 = 16)
assert H * W == N

spatial = patches.permute(0, 2, 1).reshape(B, D, H, W)
print(f"patch tokens:  {patches.shape}")   # [1, 256, 1024] for ViT-L/14
print(f"spatial map:   {spatial.shape}")   # [1, 1024, 16, 16]

## Notes for RadHarmony integration

```
# Zero-shot / linear probe with projected features:
model_call : lambda m, x: F.normalize(m.get_image_features(pixel_values=x), dim=-1)
embed_dim  : 512 (B) / 768 (L)

# Linear probe with raw ViT CLS (often better):
model_call : lambda m, x: m.vision_model(pixel_values=x).pooler_output
embed_dim  : 768 (ViT-B) / 1024 (ViT-L)

transform  : CLIPProcessor — resize/crop/normalize automatically
             Use EncoderPreprocessTransform.from_huggingface(keys=["img"], processor=processor)
pool       : None (in both cases)
```

**CLIP normalization** is `mean=(0.48145466, 0.4578275, 0.40821073)` —
distinct from ImageNet (0.485/0.456/0.406) and SigLIP (0.5/0.5/0.5). Always
use `CLIPProcessor`; don't borrow from another backbone.

**Comparison value**: Running vanilla CLIP alongside BiomedCLIP / CheXzero on
the same VinDr split quantifies exactly how much domain pretraining helps.

**Segmentation mode** (`output_keys={"img", "mask"}`):
- Patch tokens: `model.vision_model(pixel_values=x).last_hidden_state[:, 1:]`
- Spatial map: reshape to `[B, 1024, 16, 16]` for ViT-L/14 at 224px (224/14=16 → 16×16)
- `D_vision` = 1024 (ViT-L) / 768 (ViT-B) — pre-projection, larger than the 512/768 joint-space dim

In [ ]:
CXR_PATH = "/path/to/cxr.jpg"

img = Image.open(CXR_PATH).convert("RGB")
inp = processor(images=img, return_tensors="pt")

with torch.no_grad():
    feats = F.normalize(model.get_image_features(**inp), dim=-1)   # [1, 768]

text_inputs = processor(text=prompts, return_tensors="pt", padding=True)
with torch.no_grad():
    tnorm = F.normalize(model.get_text_features(**text_inputs), dim=-1)

logits = model.logit_scale.exp() * (feats @ tnorm.T).squeeze(0)
probs  = logits.softmax(dim=0)
for p, prob in sorted(zip(prompts, probs.tolist()), key=lambda x: -x[1]):
    print(f"{prob:.3f}  {p}")

## Notes for RadHarmony integration

```
# Zero-shot / linear probe with projected features:
model_call : lambda m, x: F.normalize(m.get_image_features(pixel_values=x), dim=-1)
embed_dim  : 512 (B) / 768 (L)

# Linear probe with raw ViT CLS (often better):
model_call : lambda m, x: m.vision_model(pixel_values=x).pooler_output
embed_dim  : 768 (ViT-B) / 1024 (ViT-L)

transform  : CLIPProcessor — resize/crop/normalize automatically
             Use EncoderPreprocessTransform.from_huggingface(keys=["img"], processor=processor)
pool       : None (in both cases)
```

**CLIP normalization** is `mean=(0.48145466, 0.4578275, 0.40821073)` —
distinct from ImageNet (0.485/0.456/0.406) and SigLIP (0.5/0.5/0.5). Always
use `CLIPProcessor`; don't borrow from another backbone.

**Comparison value**: Running vanilla CLIP alongside BiomedCLIP / CheXzero on
the same VinDr split quantifies exactly how much domain pretraining helps.